In [27]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score



In [28]:
feat = pd.read_csv('credit_card_featured.csv')

In [29]:
raw_cols = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT']
payment_ratio_cols = ['PAY_RATIO_SEP', 'PAY_RATIO_AUG',
       'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY']

In [30]:
X = feat
y = X['default.payment.next.month']
X = feat[raw_cols + payment_ratio_cols]

In [31]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [32]:
logreg = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0, add_indicator=True)),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

In [34]:
model = XGBClassifier(eval_metric = 'logloss', random_state=0)

In [35]:
probs_by_model = {}
for name, m in [('logreg', logreg), ('xgb', model)]:
    probs_by_model[name] = cross_val_predict(m, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

# xgb is the stronger model (see 02_baseline.ipynb / 04_modeling.ipynb), so it drives
# the threshold/cost analysis below; logreg's predictions are kept in probs_by_model
# for reference.
probs = probs_by_model['xgb']

In [36]:
print(probs)

[0.01162645 0.01033933 0.11446184 ... 0.115658   0.53880477 0.61907744]


In [37]:
tn, fp, fn, tp = confusion_matrix(y_train, (probs >= 0.5).astype(int)).ravel()
print(tn, fp, fn, tp)

17553 1138 3371 1938


In [38]:
COST_FP = 50

In [39]:
def sweep(cost_fn):
    COST_FN = cost_fn
    rows = []
    for t in np.arange(0.01, 0.96, 0.01):
        tn, fp, fn, tp = confusion_matrix(y_train, (probs >= t).astype(int)).ravel()
        rows.append({
            'threshold': t,
            'flagged': tp + fp,
            'fn': fn,
            'fp': fp,
            'cost': fn * COST_FN + fp * COST_FP,
            'recall': tp / (tp + fn),
            'precision': tp / (tp + fp) if (tp + fp) > 0 else 0
        })

    df_sweep = pd.DataFrame(rows)

    best = df_sweep.loc[df_sweep['cost'].idxmin()]
    return df_sweep, best

In [40]:
table, best = sweep(1000)
print(best)


threshold         0.010000
flagged       23103.000000
fn               40.000000
fp            17834.000000
cost         931700.000000
recall            0.992466
precision         0.228066
Name: 0, dtype: float64


In [41]:
table_5to1, best_5to1 = sweep(250)
print(best_5to1)

threshold         0.150000
flagged       10396.000000
fn             1464.000000
fp             6551.000000
cost         693550.000000
recall            0.724242
precision         0.369854
Name: 14, dtype: float64


In [42]:
n_train = len(y_train)
cap_rows = []
for pct in [0.05, 0.10, 0.20]:
    target = pct * n_train
    row = table.loc[(table['flagged'] - target).abs().idxmin()]
    cap_rows.append({
        'capacity_pct': pct,
        'threshold': round(row['threshold'], 2),
        'flagged': int(row['flagged']),
        'recall': round(row['recall'], 3),
        'precision': round(row['precision'], 3),
    })

capacity_table = pd.DataFrame(cap_rows)
print(capacity_table)

   capacity_pct  threshold  flagged  recall  precision
0          0.05       0.77     1204   0.165      0.726
1          0.10       0.59     2394   0.301      0.667
2          0.20       0.35     4715   0.478      0.538


In [43]:
cols = raw_cols + payment_ratio_cols
model.fit(X_train[cols], y_train)
pred_prob = model.predict_proba(X_test[cols])[:, 1]

print(roc_auc_score(y_test, pred_prob))
print(average_precision_score(y_test, pred_prob))
print(confusion_matrix(y_test, (pred_prob >= 0.59).astype(int)))

0.7596719179638485
0.5284749119799532
[[4472  201]
 [ 943  384]]


Under the assumed cost ratio of 20:1, the cost-minimizing threshold falls below 0.01, flagging nearly the entire portfolio. This is a property of the assumption, not a useful policy: when a miss is 20x a false alarm and 22% of customers default, blanket intervention beats any selective strategy. Only at a 5:1 ratio does a selective threshold (0.15) emerge.

Contacting the model's top-ranked 10% of customers identifies 30% of all defaulters — three times what random selection would achieve — with two-thirds of contacts reaching a customer who does go on to default.

### Without Categorical values

In [44]:
protected = ['SEX', 'EDUCATION', 'MARRIAGE']
cols = raw_cols + payment_ratio_cols
cols_no_protected = [c for c in cols if c not in protected]

In [45]:
for label, c in [('with protected', cols), ('without protected', cols_no_protected)]:
    auc = cross_val_score(model, X_train[c], y_train, cv=cv, scoring='roc_auc')
    print('%s: AUC %.4f +/- %.4f' % (label, auc.mean(), auc.std()))

with protected: AUC 0.7610 +/- 0.0041
without protected: AUC 0.7567 +/- 0.0041


In [46]:
pred_np = cross_val_predict(model, X_train[cols_no_protected], y_train,
                             cv=cv, method='predict_proba')[:, 1]

In [47]:
fair = pd.DataFrame({
    'y': y_train.values,
    'pred_with': (probs >= 0.59).astype(int),
    'pred_without': (pred_np >= 0.59).astype(int),
    'SEX': X_train['SEX'].values,
    'EDUCATION': X_train['EDUCATION'].values,
    'MARRIAGE': X_train['MARRIAGE'].values,
})

In [48]:
defaulters = fair[fair['y'] == 1]
print(defaulters.groupby('SEX')[['pred_with', 'pred_without']].mean())
print(defaulters.groupby('EDUCATION')[['pred_with', 'pred_without']].mean())
print(defaulters.groupby('MARRIAGE')[['pred_with', 'pred_without']].mean())

     pred_with  pred_without
SEX                         
1     0.315674      0.309982
2     0.289256      0.301488
           pred_with  pred_without
EDUCATION                         
1           0.261482      0.258420
2           0.325633      0.328241
3           0.304752      0.325413
4           0.000000      0.083333
          pred_with  pred_without
MARRIAGE                         
1          0.314873      0.309335
2          0.287929      0.302326
3          0.277778      0.263889
